In [1]:
# 必要なモジュールをインポート
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletionToolParam
from tavily import TavilyClient

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# tavily検索用APIキーの取得
TAVILY_API_KEY = os.environ['TAVILY_API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [2]:
# 検索結果を返す関数の作成
def get_search_result(question):
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(question)
    return json.dumps({"result": response["results"]})

In [3]:
# テスト用コード
ret = get_search_result("東京駅のイベントを教えて")
json.loads(ret)

{'result': [{'url': 'https://media.jreast.co.jp/tags/9756',
   'title': '東京駅イベントに関する最新情報・おすすめ記事 | JREメディア',
   'content': ':   「東京駅イベント」でタグ付けされた記事一覧です。JREメディアには「東京駅イベント」に関する記事やご案内、便利な情報が8件掲載されています。. # 東京駅で群馬溫泉文化旅遊！温泉王国ぐんま体感イベントを開催. 2026年2月20日（金）～22日（日）に、JR東日本東京駅B1「スクエア ゼロ」で開催される「群馬溫泉... # 東京駅で開催！青森・北海道道南産直市＠スクエアゼロ. 「青森県・函館観光キャンペーン」期間中、JR東日本クロスステーションは、エキナカ地域フェア「青森・北海道... # 東京駅で開催中の春イベント「TOKYO EASTER & SWEETS」をレポート！. 2025年4月20日(日)まで東京駅特設会場で開催中の「TOKYO EASTER&SWEETS」。対象店... # 東京駅発春イベント「TOKYO EASTER & SWEETS」開催！. JR東京駅B1改札内イベントスペース「スクエア ゼロ」にて、春の訪れを祝福するお祭り”イースター”をテー... # クリスマスは謎解きを楽しもう♪『東京駅サンタ謎～110年目のプレゼント～』開催！. # 東京駅開業110周年記念イベント12月開催決定！そのイベントの内容とは!? 東京駅は2024年12月に開業110周年を迎えます。東京駅は、東北・上越・北陸新幹線や東海道新幹線の発着... # 東京駅で北陸フェア開催！北陸の”おいしい”が大集合【のもの東京】. 北陸デスティネーションキャンペーンに合わせて、2024年11月19日(火)～2024年12月2日(月)ま... # 東京駅で長野フェア開催！長野の“おいしい”が大集合【のもの東京】. “ココロうごく体験”をお届けする夏の信州観光キャンペーンに合わせて、2024年9月17日(火)～2024... ## アクセスランキング. ### 平日限定で50％割引！新幹線や特急列車に乗っておトクに平日旅♪. ### 東京駅でしか買えない限定お土産25選【2026年最新版】. 特急「踊り子」チケットレス特急券が半額！平

In [4]:
# ツール定義
def define_tools():
    print("------define_tools(ツール定義)------")
    return [
        ChatCompletionToolParam({
            "type": "function",
            "function": {
                "name": "get_search_result",
                "description": "最近一ヵ月のイベント開催予定などネット検索が必要な場合に、質問文の検索結果を取得する",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string", "description": "質問文"},
                    },
                    "required": ["question"],
                },
            },
        })
    ]

In [5]:
# 言語モデルへの質問を行う関数
def ask_question(question, tools):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": question}],
        tools=tools,
        tool_choice="auto",
    )
    return response

In [6]:
# ツール呼び出しが必要な場合の処理を行う関数
def handle_tool_call(response, question):
    # 関数の実行と結果取得
    tool = response.choices[0].message.tool_calls[0]
    function_name = tool.function.name
    arguments = json.loads(tool.function.arguments)
    function_response = globals()[function_name](**arguments)

    # 関数の実行結果をmessagesに加えて再度言語モデルを呼出
    response_after_tool_call = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": question},
            response.choices[0].message,
            {
                "tool_call_id": tool.id,
                "role": "tool",
                "content": function_response,
            },
        ],
    )
    return response_after_tool_call

In [7]:
# ユーザーからの質問を処理する関数
def process_response(question, tools):
    response = ask_question(question, tools)

    if response.choices[0].finish_reason == 'tool_calls':
        # ツール呼出の場合
        final_response = handle_tool_call(response, question)
        return final_response.choices[0].message.content.strip()
    else:
        # 言語モデルが直接回答する場合
        return response.choices[0].message.content.strip()

In [8]:
tools = define_tools()

# 言語モデルが直接回答できる質問
question = "東京都と沖縄県はどちらが広いですか？"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
東京都と沖縄県の面積を比較すると、沖縄県の方が広いです。

- 東京都の面積は約2,194平方キロメートルです。
- 沖縄県の面積は約2,271平方キロメートルです。

したがって、沖縄県の方が東京都よりも広いということになります。


In [9]:
tools = define_tools()

# ツール呼出が必要な質問
question = "東京駅のイベントについて、最近1ヶ月以内の検索結果を教えてください"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
最近1ヶ月以内に東京駅で行われるイベントに関する情報をいくつかご紹介します。

1. **[東京駅六周年祭](https://www.walkerplus.com/event_list/ar0313/sc309880d/)**
   - 開催日: 2026年4月19日 
   - 内容: アートとリストランの祭典。さまざまなアーティストによる作品展示が行われます。

2. **[POP UP SHOP in 東京駅](https://www.tokyoeki-1bangai.co.jp/event/)**
   - 開催日: 2026年4月10日 - 4月30日
   - 内容: TVアニメ「ダイアのA act」のPOP UP SHOPがオープンします。

3. **[東京駅限定イベント](https://bestcalendar.jp/events/%E6%9D%B1%E4%BA%AC%E9%A7%85)**
   - 開催日: 2026年4月27日 - 5月6日
   - 内容: 地元の特産をテーマにしたイベントで、歴史的な展示や地元の飲食が楽しめます。

4. **[東京駅近辺のイベント情報](https://www.enjoytokyo.jp/event/list/area1306/)**
   - 内容: 東京駅周辺では、様々な季節イベントやキャンペーンが行われており、詳細なスケジュールは公式サイトから確認できます。

これらのイベントは、参加や観覧に際して事前に予約や問い合わせが必要な場合がありますので、訪れる前に詳細を確認することをおすすめします。


In [10]:
# チャットボットへの組み込み
tools = define_tools()

messages=[]

while(True):
    # ユーザーからの質問を受付
    question = input("メッセージを入力:")
    # 質問が入力されなければ終了
    if question.strip()=="":
        break
    display(f"質問:{question}")

    # メッセージにユーザーからの質問を追加
    messages.append({"role": "user", "content": question.strip()})
    # やりとりが8を超えたら古いメッセージから削除
    if len(messages) > 8:
        del_message = messages.pop(0)

    # 言語モデルに質問
    response_message = process_response(question, tools)

    # メッセージに言語モデルからの回答を追加
    print(response_message, flush=True)
    messages.append({"role": "assistant", "content": response_message})

print("\n---ご利用ありがとうございました！---")

------define_tools(ツール定義)------


'質問:東北6県は？'

東北地方は日本の北部に位置しており、以下の6つの県から構成されています。

1. 青森県（あおもりけん）
2. 岩手県（いわてけん）
3. 宮城県（みやぎけん）
4. 秋田県（あきたけん）
5. 山形県（やまがたけん）
6. 福島県（ふくしまけん）

これらの県は、自然が豊かで歴史的な名所や文化も多い地域です。


'質問:青森県のお土産について検索した結果を教えて'

青森県のお土産には、以下のような特産品やお菓子が人気です：

1. **りんご**: 青森県は日本一のりんごの生産地で、様々な品種があります。新鮮なりんごや、りんごを使った加工品も多くお土産におすすめです。

2. **いちご煮**: いちご煮は、青森県特有の海の幸を使ったご飯のお供。特に貝類を基にした甘辛い煮物で、贈り物に人気です。

3. **南部せんべい**: 青森の伝統的なせんべいで、小麦粉を主成分とし、様々な味があります。お土産として非常に人気があります。

4. **豆腐料理**: 特に「すかたん」などの豆腐菓子や、青森牛乳を使った濃厚な豆腐スイーツも人気です。

5. **ねぶた祭り関連商品**: ねぶた祭りに関連するお土産も人気で、ちょっとしたお土産に最適です。

6. **地酒**: 青森県は多くの酒蔵があり、地元の米を使用した日本酒が豊富です。お酒好きの方へのお土産としておすすめです。

7. **スイーツ**: 特に「ダックワーズ」や、青森県産の材料を使用したお菓子が人気です。

詳しい情報や具体的な商品については、青森県の観光ウェブサイトやお土産専用のオンラインストアをチェックするのも良いでしょう。

---ご利用ありがとうございました！---
